In [ ]:
import numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors

# ---------------- utilities ----------------
def _to_days(s):
    t = pd.to_datetime(s, errors="coerce").view("int64") // 86400_000_000_000
    return t.astype("int64")

def _estimate_alpha(x, y, t_days):
    P = np.c_[x, y]
    if len(P) < 3:
        return 1.0
    k = min(6, len(P))
    nn = NearestNeighbors(n_neighbors=k).fit(P)
    d = nn.kneighbors(return_distance=True)[0][:,1:]
    r_s = np.median(d) if d.size else 1.0
    ut  = np.unique(t_days)
    gaps = np.diff(ut)
    r_t = np.median(gaps[gaps>0]) if np.any(gaps>0) else 1.0
    return (r_s / r_t)**2

def _weights(dist, kind="gaussian", power=2.0, eps=None):
    if kind == "gaussian":
        if eps is None:
            nz = dist[dist>1e-12]
            eps = np.median(nz) if nz.size else 1.0
        u = dist / eps
        return np.exp(-(u*u))
    # idw
    return 1.0 / np.maximum(dist, 1e-6)**power

# ---------------- main ----------------
def fill_ibd_nans(ibd,
                  k=32,
                  kernel="gaussian",   # "gaussian" or "idw"
                  power=2.0,           # only for idw
                  eps=None,            # None → auto from neighbor distances
                  alpha=None,          # None → auto from ranges
                  same_site_only=False,
                  site_col="CodeSite_SamplingOperations",
                  same_xy_tol=0.5):
    """
    Fill NaNs in 'IBD' using k-NN weighting in (x,y, sqrt(alpha)*t).
    Columns required: Date_SamplingOperation, Longitude_Lambert93, Latitude_Lambert93, IBD.
    """
    df = ibd.copy()
    df["_x"] = df["Longitude_Lambert93"].astype(float)
    df["_y"] = df["Latitude_Lambert93"].astype(float)
    df["_t"] = _to_days(df["Date_SamplingOperation"])

    mask_known = df["IBD"].notna()
    mask_query = df["IBD"].isna() & df[["_x","_y","_t"]].notna().all(1)
    if mask_query.sum() == 0 or mask_known.sum() < 3:
        return df

    xk = df.loc[mask_known, "_x"].to_numpy()
    yk = df.loc[mask_known, "_y"].to_numpy()
    tk = df.loc[mask_known, "_t"].to_numpy()
    zk = df.loc[mask_known, "IBD"].to_numpy().astype(float)

    if alpha is None:
        alpha = _estimate_alpha(xk, yk, tk)

    Zk = np.c_[xk, yk, np.sqrt(alpha)*tk]
    k_eff = min(k, len(Zk))
    nbrs = NearestNeighbors(n_neighbors=k_eff).fit(Zk)

    xq = df.loc[mask_query, "_x"].to_numpy()
    yq = df.loc[mask_query, "_y"].to_numpy()
    tq = df.loc[mask_query, "_t"].to_numpy()
    Zq = np.c_[xq, yq, np.sqrt(alpha)*tq]

    # Optional restriction: use only same-site neighbors
    if same_site_only:
        # by site id if available, else by XY tolerance
        out = np.full(len(Zq), np.nan, float)
        if site_col in df.columns:
            site_k = df.loc[mask_known, site_col].to_numpy()
            site_q = df.loc[mask_query, site_col].to_numpy()
            from collections import defaultdict
            bucket = defaultdict(list)
            for i,s in enumerate(site_k): bucket[s].append(i)
            for j, (zq, s) in enumerate(zip(Zq, site_q)):
                pool = np.array(bucket.get(s, []), dtype=int)
                if pool.size == 0:
                    # fallback to global neighbors
                    dist, idx = nbrs.kneighbors(zq.reshape(1,-1), return_distance=True)
                    dist, idx = dist[0], idx[0]
                else:
                    D = np.linalg.norm(Zk[pool] - zq, axis=1)
                    take = min(k_eff, len(pool))
                    order = np.argpartition(D, take-1)[:take]
                    idx = pool[order]
                    dist = D[order]
                if dist[0] < 1e-12:
                    out[j] = zk[idx[0]]
                else:
                    w = _weights(dist, kind=kernel, power=power, eps=eps)
                    out[j] = np.sum(w * zk[idx]) / np.sum(w)
    else:
        dist, idx = nbrs.kneighbors(Zq, return_distance=True)
        # exact matches
        exact = dist[:,0] < 1e-12
        out = np.empty(len(Zq), float)
        out[exact] = zk[idx[exact,0]]
        # rest
        rest = ~exact
        if np.any(rest):
            D = dist[rest]
            I = idx[rest]
            if kernel == "gaussian" and eps is None:
                nz = D[D>1e-12]
                eps_auto = np.median(nz) if nz.size else 1.0
            else:
                eps_auto = eps
            W = _weights(D, kind=kernel, power=power, eps=eps_auto)
            Y = zk[I]
            num = (W * Y).sum(1)
            den = W.sum(1)
            out[rest] = num / den

    df.loc[mask_query, "IBD"] = out
    return df  # if you need alpha, return (df, alpha)


In [ ]:
# Gaussian, auto alpha, k=32
ibd_filled = fill_ibd_nans(ibd, kernel="gaussian", k=32)

# Make time stronger: 1 year ~ 20 km
alpha = (20_000/365.0)**2
ibd_filled = fill_ibd_nans(ibd, kernel="gaussian", k=32, alpha=alpha)

# Same-site-only fill if you have a site column
ibd_filled = fill_ibd_nans(ibd, kernel="gaussian", same_site_only=True,
                           site_col="CodeSite_SamplingOperations")
